# Lab: Gradient Descent and Regularization

**Learning Objectives:**
- Understand how gradient descent optimizes models
- Explore different batching strategies and their bias-variance tradeoffs
- Learn how regularization improves generalization
- Implement early stopping as a regularization technique

---

## Setup

First, let's import the libraries we'll need and set up our plotting style.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 4)

## Part 1: Understanding Gradient Descent

Let's start by visualizing how gradient descent finds the minimum of a loss function.

### Step 1.1: Generate and Visualize Data

In [ ]:
def generate_polynomial_data(n_samples=100, noise=0.3):
    """
    Generate synthetic data with polynomial relationship + noise
    
    TODO: Complete this function
    1. Create X as evenly spaced points from -3 to 3
    2. Create y using: y = 0.5*X^2 - 2*X + 1 + noise
    3. The noise should be Gaussian with mean 0 and std=noise parameter
    """
    # YOUR CODE HERE
    X = None  # HINT: use np.linspace(-3, 3, n_samples)
    y = None  # HINT: 0.5 * X**2 - 2*X + 1 + np.random.normal(0, noise, n_samples)
    
    return X.reshape(-1, 1), y

# Generate data - FEWER samples to create overfitting scenario
# With 60 samples and degree-25 polynomial, we'll be severely overparameterized!
X, y = generate_polynomial_data(n_samples=60, noise=1.0)

# Split into train/validation/test (49%/21%/30% split)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.3, random_state=123)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples: {len(X_test)}")

# TODO: Visualize the data
# Create a scatter plot with different colors for train/val/test sets
# YOUR CODE HERE


### Step 1.2: Implement Gradient Descent

Now let's implement gradient descent for polynomial regression. 

**Recall:**
- Loss function (MSE): $L(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^{n}(y_i - \mathbf{w}^T\mathbf{x}_i)^2$
- Gradient: $\nabla L(\mathbf{w}) = -\frac{2}{n}\mathbf{X}^T(\mathbf{y} - \mathbf{X}\mathbf{w})$
- Update rule: $\mathbf{w} \leftarrow \mathbf{w} - \alpha \nabla L(\mathbf{w})$

In [ ]:
def create_polynomial_features(X, degree):
    """Create polynomial features up to specified degree"""
    poly = PolynomialFeatures(degree=degree, include_bias=True)
    return poly.fit_transform(X)

def gradient_descent(X, y, learning_rate=0.01, n_iterations=1000, batch_size=None):
    """
    Gradient descent for linear regression
    
    TODO: Complete this function
    1. Initialize weights to zeros
    2. For each iteration:
       a. Select batch (all data if batch_size=None, otherwise random sample)
       b. Compute predictions: X_batch @ weights
       c. Compute gradient: -2 * X_batch.T @ (y_batch - predictions) / batch_size
       d. Update weights: weights = weights - learning_rate * gradient
       e. Compute and store loss on full dataset
    3. Return weights and loss history
    
    Parameters:
    - X: features (n_samples, n_features)
    - y: targets (n_samples,)
    - learning_rate: step size
    - n_iterations: number of update steps
    - batch_size: if None, use full batch GD. Otherwise, use mini-batch GD
    """
    n_samples, n_features = X.shape
    
    # YOUR CODE HERE
    weights = None  # HINT: np.zeros(n_features)
    loss_history = []
    
    for iteration in range(n_iterations):
        # Select batch
        if batch_size is None:
            # Full batch gradient descent
            X_batch = None  # YOUR CODE
            y_batch = None  # YOUR CODE
        else:
            # Mini-batch gradient descent
            indices = None  # HINT: np.random.choice
            X_batch = None  # YOUR CODE
            y_batch = None  # YOUR CODE
        
        # Compute predictions
        predictions = None  # YOUR CODE
        
        # Compute gradient
        gradient = None  # YOUR CODE (use formula from docstring)
        
        # Update weights
        weights = None  # YOUR CODE
        
        # Compute loss on full dataset (for monitoring)
        full_predictions = None  # YOUR CODE
        loss = None  # YOUR CODE: MSE
        loss_history.append(loss)
    
    return weights, loss_history

### Step 1.3: Train and Visualize Results

In [ ]:
# Create polynomial features (degree 2)
degree = 2
X_train_poly = create_polynomial_features(X_train, degree)
X_val_poly = create_polynomial_features(X_val, degree)

# TODO: Run gradient descent
# Call your gradient_descent function with appropriate parameters
# Try learning_rate=0.01, n_iterations=500
# YOUR CODE HERE
weights, loss_history = None

# TODO: Create two subplots side-by-side
# Left plot: loss_history over iterations
# Right plot: scatter plot of training data + fitted model line
# YOUR CODE HERE


**Question 1.1:** What do you observe about the convergence? Does the loss decrease smoothly?

*Your answer here:*

---

## Part 2: Batch Size and Bias-Variance Tradeoff

Different batching strategies have different properties:
- **Batch GD**: Uses all data points → Low variance, can get stuck
- **Stochastic GD (batch_size=1)**: Uses one point → High variance, more exploration
- **Mini-batch GD**: Middle ground → Balance

### Step 2.1: Compare Different Batch Sizes

In [ ]:
# TODO: Compare different batch sizes
# Test these configurations: Full Batch (None), Mini-batch (32), Mini-batch (8), Stochastic (1)
# For each, run gradient descent and plot the loss curves

batch_configs = [
    ('Full Batch', None),
    ('Mini-batch (32)', 32),
    ('Mini-batch (8)', 8),
    ('Stochastic (1)', 1)
]

# YOUR CODE HERE
# HINT: Loop through batch_configs, run gradient_descent for each, and plot all on same figure


**Question 2.1:** Which batch size converges fastest? Which is most stable?

*Your answer here:*

**Question 2.2:** Why does stochastic GD (batch_size=1) have such noisy loss curves?

*Your answer here:*

---

## Part 3: Regularization and Generalization

Regularization adds a penalty to prevent overfitting:

**L2 (Ridge):** $L(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^{n}(y_i - \mathbf{w}^T\mathbf{x}_i)^2 + \lambda||\mathbf{w}||_2^2$

**L1 (Lasso):** $L(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^{n}(y_i - \mathbf{w}^T\mathbf{x}_i)^2 + \lambda||\mathbf{w}||_1$

### Step 3.1: Implement Gradient Descent with Regularization

In [ ]:
def gradient_descent_with_regularization(
    X, y, X_val, y_val, 
    learning_rate=0.01, 
    n_iterations=1000,
    lambda_reg=0.0,
    reg_type='l2'
):
    """
    Gradient descent with L1 or L2 regularization
    
    TODO: Complete this function
    1. Start with standard gradient descent
    2. Add regularization gradient:
       - L2: gradient += 2 * lambda_reg * weights (don't regularize bias at index 0)
       - L1: gradient += lambda_reg * sign(weights) (don't regularize bias at index 0)
    3. Track both train and validation loss
    
    Parameters:
    - lambda_reg: regularization strength
    - reg_type: 'l1' or 'l2'
    """
    n_samples, n_features = X.shape
    
    # YOUR CODE HERE
    weights = None
    train_loss_history = []
    val_loss_history = []
    
    for iteration in range(n_iterations):
        # Predictions
        predictions = None  # YOUR CODE
        
        # Compute gradient of MSE
        gradient = None  # YOUR CODE
        
        # TODO: Add regularization gradient
        # Remember: don't regularize the bias term (index 0)
        if lambda_reg > 0:
            if reg_type == 'l2':
                # L2: gradient is 2*lambda*w
                # YOUR CODE HERE
                pass
            elif reg_type == 'l1':
                # L1: gradient is lambda*sign(w)
                # YOUR CODE HERE
                pass
        
        # Update weights
        weights = None  # YOUR CODE
        
        # Track losses (without regularization term for fair comparison)
        train_loss = None  # YOUR CODE: MSE on training set
        val_loss = None    # YOUR CODE: MSE on validation set
        train_loss_history.append(train_loss)
        val_loss_history.append(val_loss)
    
    return weights, train_loss_history, val_loss_history

### Step 3.2: Experiment with High-Degree Polynomial (Overfitting Scenario)

In [ ]:
# Use high-degree polynomial to create overfitting scenario
# We'll fit a degree-25 polynomial to data generated from a degree-2 function
# With only ~29 training samples and 26 features, we'll have severe overparameterization!
# This is the PERFECT recipe for overfitting!
high_degree = 25
X_train_high = create_polynomial_features(X_train, high_degree)
X_val_high = create_polynomial_features(X_val, high_degree)
X_test_high = create_polynomial_features(X_test, high_degree)

print(f"Training samples: {len(X_train)}")
print(f"Number of features: {X_train_high.shape[1]}")
print(f"Feature/Sample ratio: {X_train_high.shape[1]/len(X_train):.2f}")
print("(Ratio > 0.5 means overparameterized → overfitting expected!)\n")

# IMPORTANT: Standardize features to prevent numerical instability
# High-degree polynomials create very large values (X^25) that cause NaN in gradient descent
# TODO: Import StandardScaler and apply it to X_train_high, X_val_high, X_test_high
# HINT: Don't scale the bias column (index 0), only columns 1 onwards
# YOUR CODE HERE
from sklearn.preprocessing import StandardScaler
scaler = None  # Create StandardScaler(with_mean=False, with_std=True)
# Fit on X_train_high[:, 1:] and transform all three datasets

# TODO: Try different regularization strengths
# Test lambda values: [0.0, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
# For each lambda:
#   1. Train with L2 regularization (use learning_rate=0.01, n_iterations=5000)
#   2. Evaluate on test set
#   3. Calculate the gap: val_loss - train_loss
#      → POSITIVE gap = overfitting (val > train)
#      → NEGATIVE gap = possible underfitting (val < train)
#   4. Plot the fitted curve (remember to scale X_plot features too!)
# Create a 2x4 grid of subplots showing results for each lambda

lambda_values = [0.0, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]

# YOUR CODE HERE
# HINT: Loop through lambda_values, train models, and create visualization
# HINT: For plotting, also show the true function: y = 0.5*X^2 - 2*X + 1 (in green dashed line)
# HINT: Color-code subplot titles: red if gap > 2, orange if gap > 0.5, green otherwise

# EXPECTED RESULTS:
# λ=0.0: You should see train loss ~ 0.9 but val loss ~ 5.7 (HUGE gap = severe overfit!)
# λ=0.1-0.2: Optimal zone where val loss is minimized
# λ=1.0: Both losses high (underfitting)


**Question 3.1:** What happens to the fitted curve as you increase λ? Why?

*Your answer here:*

**Question 3.2:** Which λ value gives the best test performance? Does it have the lowest training loss?

*Your answer here:*

---

### Step 3.3: Compare L1 vs L2 Regularization

In [ ]:
# TODO: Compare L1 and L2 at the same lambda value
# Train two models with lambda=0.1, one with L1 and one with L2
# Create visualizations comparing:
#   1. Weight magnitudes
#   2. Training curves
#   3. Number of near-zero weights (sparsity)

lambda_reg = 0.1

# YOUR CODE HERE


**Question 3.3:** Which regularization method (L1 or L2) produces more sparse weights? Why is this useful?

*Your answer here:*

---

## Part 4: Early Stopping

Early stopping is implicit regularization - we stop training when validation performance stops improving.

### Step 4.1: Implement Early Stopping

In [ ]:
def gradient_descent_with_early_stopping(
    X, y, X_val, y_val,
    learning_rate=0.01,
    max_iterations=5000,
    patience=50,
    min_delta=1e-4
):
    """
    Gradient descent with early stopping
    
    TODO: Complete this function
    1. Track best_val_loss and best_weights
    2. Use a patience counter
    3. If validation loss improves by at least min_delta, reset counter
    4. If counter reaches patience, stop training and return best weights
    
    Parameters:
    - patience: number of iterations to wait for improvement
    - min_delta: minimum change to qualify as improvement
    """
    n_samples, n_features = X.shape
    
    # YOUR CODE HERE
    weights = None
    best_weights = None
    best_val_loss = float('inf')
    patience_counter = 0
    
    train_loss_history = []
    val_loss_history = []
    
    for iteration in range(max_iterations):
        # Standard gradient descent step
        # YOUR CODE HERE
        
        # Compute losses
        # YOUR CODE HERE
        
        # Early stopping check
        # YOUR CODE HERE
        # HINT: Check if val_loss < best_val_loss - min_delta
        
        # If patience exceeded, break
        # YOUR CODE HERE
    
    return best_weights, train_loss_history, val_loss_history, iteration

### Step 4.2: Compare With and Without Early Stopping

In [ ]:
# TODO: Train two models on the high-degree polynomial:
# 1. Without early stopping (3000 iterations, no regularization)
# 2. With early stopping (patience=100)
# 
# Create visualizations comparing:
#   - Training curves (mark where early stopping occurred)
#   - Fitted curves
#   - Test set performance

# YOUR CODE HERE


**Question 4.1:** Does early stopping help prevent overfitting? How does test performance compare?

*Your answer here:*

---

## Part 5: Comprehensive Comparison

### Step 5.1: Compare All Techniques

In [ ]:
# TODO: Compare these approaches on the high-degree polynomial:
# 1. No Regularization
# 2. L2 (λ=0.1)
# 3. L1 (λ=0.1)
# 4. Early Stopping
# 5. L2 + Early Stopping
#
# Create a summary table with:
# - Train Loss
# - Val Loss
# - Test Loss
# - Train-Val Gap (measure of overfitting)

# YOUR CODE HERE
import pandas as pd

# Store results in a list of dictionaries, then create DataFrame


**Question 5.1:** Which technique(s) work best? Why?

*Your answer here:*

**Question 5.2:** Can combining regularization techniques improve performance further?

*Your answer here:*

---

## Summary

In this lab, you learned:

1. **Gradient Descent Basics**: How iterative optimization finds model parameters

2. **Batch Size Tradeoffs**:
   - Full batch: Stable but slow
   - Stochastic: Fast but noisy
   - Mini-batch: Best of both worlds

3. **Regularization for Generalization**:
   - L2 (Ridge): Shrinks all weights smoothly
   - L1 (Lasso): Induces sparsity
   - Prevents overfitting

4. **Early Stopping**: Implicit regularization by stopping before overfitting

**Key Takeaway**: The goal isn't to minimize training loss, but to build models that generalize well to new data!

---

### Additional Resources:
- [An overview of gradient descent optimization algorithms](https://ruder.io/optimizing-gradient-descent/)
- [Deep Learning Book - Chapter 7 (Regularization)](https://www.deeplearningbook.org/contents/regularization.html)
- [CS231n: Optimization Notes](http://cs231n.github.io/optimization-1/)